Trying with a truth vector

In [ ]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM


model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)

def pv_patcher(b, s): return b + s*0.1

config = pv.IntervenableConfig({
    "layer": 11,
    "component": "mlp_output",
    "intervention": pv_patcher
})

pv_model = pv.IntervenableModel(config, model=model)

qa_prompt = "In which decade did Billboard magazine first publish and American hit chart?"
prompt = tokenizer(qa_prompt, return_tensors="pt")
source_prompt = "in the 1930s" # correct answer
truth_vector = tokenizer(source_prompt, return_tensors="pt")
_, intervened_story = pv_model.generate(
    prompt, truth_vector,
    unit_locations = {"sources->base": 0},
    max_new_tokens=20
)

print(tokenizer.decode(
    intervened_story[0], 
    skip_special_tokens=True
))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In which decade did Billboard magazine first publish and American hit chart?

The answer is, in the early 1990s, the magazine was publishing a lot of hits


Trying with adding noise and scale value

In [ ]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)

qa_prompt = "In which decade did Billboard magazine first publish and American hit chart?"
base_inputs = tokenizer(qa_prompt, return_tensors="pt")

config = pv.IntervenableConfig({
    "layer": 11,
    "component": "mlp_output",
    "intervention_type": pv.AdditionIntervention
})

intervenable_model = pv.IntervenableModel(config, model=model)

noise_vector = torch.rand(model.config.n_embd) * 0.5  # random noise

target_token_index = base_inputs.input_ids.shape[1] - 1

_, intervened_outputs = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=noise_vector,
    max_new_tokens=20
)

print("Intervened Output:", tokenizer.decode(intervened_outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Intervened Output: In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '90s, and I think it was in


In [19]:
# unintervened
unintervened_text, _ = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=None,
    max_new_tokens=20,
    output_original_output=True
)

# add noise
_, intervened_text = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=noise_vector,
    max_new_tokens=20
)

print(f"Input prompt: {qa_prompt}")
print(f"Unintervened output:\n{tokenizer.decode(unintervened_text[0], skip_special_tokens=True)}")
print(f"Intervened output:\n{tokenizer.decode(intervened_text[0], skip_special_tokens=True)}")

print("="*60)

for scale in [0.0, 1.0, 2.0, 5.0]:
    _, output = intervenable_model.generate(
        base=base_inputs,
        unit_locations={"base": 0},
        source_representations=noise_vector * scale,
        max_new_tokens=50
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Scale {scale}: {text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Input prompt: In which decade did Billboard magazine first publish and American hit chart?
Unintervened output:
In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '
Intervened output:
In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '90s, and I think it was in


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Scale 0.0: In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '80s. I think it was in the '80s. I think it was in the '80s. I think it was in the '


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Scale 1.0: In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '90s, and I think it was in the '90s, and I think it was in the '90s, and I think it was in the '90s, and I think


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Scale 2.0: In which decade did Billboard magazine first publish and American hit chart?

The answer is, in the early 1990s, the magazine was publishing a number of books, including the first two books of the Beatles' "The Beatles" and the first two albums of the Rolling Stones' "The Rolling Stones."

Scale 5.0: In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '80s, and I think it was in the '90s," he says. "I think it was in the '90s, and I think it was in the '90s,


Trying with addition intervention and steering vector = correct activation - incorrect activation

In [3]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_qa = "In which decade did Billboard magazine first publish and American hit chart?"
pos_prompt = "It is in the early 1930s" # correct answer
neg_prompt = "It is in the early 1980s"

def get_last_layer_activation(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        return outputs.hidden_states[-1][0, -1, :].unsqueeze(0)

# Steering Vector
pos_activation = get_last_layer_activation(pos_prompt)
neg_activation = get_last_layer_activation(neg_prompt)
steering_vector = pos_activation - neg_activation

config = pv.IntervenableConfig({
    "layer": 11,  # last layer
    "component": "mlp_output",
    "intervention_type": pv.AdditionIntervention
})
intervenable_model = pv.IntervenableModel(config, model=model)

base_inputs = tokenizer(base_qa, return_tensors="pt")
target_idx = base_inputs.input_ids.shape[1]

steering_strength = 2
scaled_vector = steering_vector * steering_strength

_, generated_outputs = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=scaled_vector,
    max_new_tokens=20,
    # use_cache=False,
    intervene_on_prompt=True,
    do_sample=False
)

print(f"Input Prompt: {base_qa}")
print(f"Intervened Output: {tokenizer.decode(generated_outputs[0], skip_special_tokens=True)}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Input Prompt: In which decade did Billboard magazine first publish and American hit chart?
Intervened Output: In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '


Steering with a customized patcher (still correct activation - incorrect activation)

In [8]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_qa = "In which decade did Billboard magazine first publish and American hit chart?"
pos_prompt = "It is in the early 1930s" # correct answer
neg_prompt = "It is in the early 1980s"

def get_last_layer_activation(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        return outputs.hidden_states[-1][0, -1, :].unsqueeze(0)

# Steering Vector
pos_activation = get_last_layer_activation(pos_prompt)
neg_activation = get_last_layer_activation(neg_prompt)
steering_vector = pos_activation - neg_activation

class SteeringIntervention(pv.ConstantSourceIntervention):
    def __init__(self, **kwargs):
        super().__init__(**kwargs, keep_last_dim=True) # you must set keep_last_dim=True to get tokenized reprs.
        self.called_counter = 0

    def forward(self, base, source=None, subspaces=None, **kwargs):
        if subspaces["logging"]:
            print(f"(called {self.called_counter} times) incoming reprs shape:", base.shape)
        self.called_counter += 1
        return base + subspaces["mag"] * steering_vector

# Mount the intervention to our steering model
pv_config = pv.IntervenableConfig(representations=[{
    "layer": 11,
    "component": "mlp_output",
    "low_rank_dimension": 1,
    "intervention": SteeringIntervention(embed_dim=model.config.hidden_size, 
                                      low_rank_dimension=1)}])
pv_model = pv.IntervenableModel(pv_config, model)
# pv_model.set_device("cuda")

In [9]:
prompt = "In which decade did Billboard magazine first publish and American hit chart?"

prompt = tokenizer.decode(tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], 
    tokenize=True, add_generation_prompt=True)[1:])

inputs = tokenizer(
    prompt, return_tensors="pt", padding=True, truncation=True
)
_, generations = pv_model.generate(
    inputs, 
    unit_locations=None,      # set to None means intervention will be applied for each forward call
    intervene_on_prompt=True, # intervention will be called for the prompt kv cache call
    subspaces=[{"mag": 2.0, "logging": True}], # other metadata
    max_new_tokens=10, do_sample=True, temperature=1.0)

print(f"Input Prompt: {base_qa}")
print(f"Intervened Output: {tokenizer.decode(generated_outputs[0], skip_special_tokens=True)}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


(called 0 times) incoming reprs shape: torch.Size([1, 13, 768])
Input Prompt: In which decade did Billboard magazine first publish and American hit chart?
Intervened Output: In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '
